# EDUAI Audio Lab · Mi voz con Kaggle GPU (Chatterbox Multilingual V3)

**Modo manual y privado:** enciendes el notebook cuando lo necesitas; ejecutas el modelo y descargas un WAV que puedes importar a EDUAI. No se abre ningún servidor ni túnel y no se requieren claves Supabase, Gemini o Hugging Face.

1. **Settings → Accelerator → GPU T4 x2**, y **Settings → Turn on internet**.
2. Mantén el notebook **Private**. No publiques audios, referencias ni los resultados. Comprueba que la celda **Verificar Internet** muestre `DNS OK` antes de instalar; si falla, activa Internet y reinicia la sesión.
3. Ejecuta las celdas en orden. La primera instalación puede tardar varios minutos. Chatterbox exige una versión de PyTorch que no coincide con la preinstalada de Kaggle; el notebook sincroniza **torch 2.6.0 + torchvision 0.21.0 + torchaudio 2.6.0 (CUDA 12.6)** y verifica los imports. Si cambió Torch, reinicia el kernel y vuelve a ejecutar las comprobaciones; la instalación correcta se detectará y no debería repetirse.
4. Sube una muestra **de tu propia voz** de 10–30 segundos al control de archivo; la convertiremos a WAV mono 24 kHz.
5. Cambia `TEXTO`, genera y descarga `eduai_voz_kaggle.wav` desde **Output**.
6. Regresa a **EDUAI → Audio Lab → Mis voces** e importa el WAV en el perfil apropiado.

La sesión de Kaggle puede finalizar en cualquier momento. Para volver a usarlo, abre de nuevo **tu notebook guardado** desde el botón de EDUAI y vuelve a ejecutar las celdas. Las dos T4 son GPU independientes: Chatterbox usa la GPU 0, con 16 GB VRAM. Este procedimiento prueba inferencia; no marca el perfil EDUAI como «clonado/ready» en ausencia de un servicio permanente.

**Privacidad:** Kaggle es un proveedor externo que procesará temporalmente la muestra al subirla. No compartas públicamente el notebook. La exportación del resultado a EDUAI es manual, para evitar exponer credenciales de administrador. Solo usar tu voz o una voz para la que tengas autorización y consentimiento.

In [ ]:
# Comprobar GPU T4 (no usar CPU para este notebook).
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Activa Settings → Accelerator → GPU T4 x2 y reinicia la sesión.")
print("GPU:", torch.cuda.get_device_name(0), "| VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

In [ ]:
# Verificar Internet ANTES de usar pip, GitHub o ModelScope.
# En Kaggle: Settings → Turn on internet (activado) y Accelerator → GPU T4 x2.
import socket
import sys
import torch

print("Python:", sys.version.split()[0])
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO DISPONIBLE")
hosts = ("pypi.org", "github.com", "modelscope.cn")
failed = []
for host in hosts:
    try:
        address = socket.getaddrinfo(host, 443, type=socket.SOCK_STREAM)[0][4][0]
        print(f"{host}: DNS OK ({address})")
    except OSError as exc:
        failed.append(host)
        print(f"{host}: SIN CONEXIÓN ({exc})")

if failed:
    raise RuntimeError(
        "Kaggle no puede resolver: " + ", ".join(failed) +
        ". Ve a Settings → Turn on internet, actívalo y reinicia la sesión. "
        "Si no aparece la opción, verifica la cuenta en https://www.kaggle.com/settings. "
        "No ejecutes pip hasta que esta celda indique DNS OK."
    )
if not torch.cuda.is_available():
    raise RuntimeError("Selecciona Settings → Accelerator → GPU T4 x2 y reinicia la sesión.")
print("Conectividad DNS comprobada; puedes iniciar la instalación.")

In [ ]:
# Instalar Chatterbox V3 y reparar compatibilidad torch/torchaudio/torchvision.
# Compatible con GPU T4 de Kaggle y Python 3.13. No descarga pesos de HF.
import os
import socket
import subprocess
import sys
import importlib.metadata as metadata

try:
    socket.getaddrinfo("pypi.org", 443, type=socket.SOCK_STREAM)
except OSError as exc:
    raise RuntimeError("Internet sin DNS. Activa Settings → Turn on internet y reinicia.") from exc

os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", HF_DATASETS_OFFLINE="1")
PINNED_TORCH = {"torch": "2.6.0", "torchvision": "0.21.0", "torchaudio": "2.6.0"}
CUDA_WHEELS = "https://download.pytorch.org/whl/cu126"

def package_version(name):
    try:
        return metadata.version(name).split("+", 1)[0]
    except metadata.PackageNotFoundError:
        return None

def run_pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

if package_version("chatterbox-tts") != "0.1.7" or not package_version("modelscope"):
    print("Instalando Chatterbox, ModelScope y widgets...")
    run_pip("chatterbox-tts==0.1.7", "modelscope>=1.20,<3", "ipywidgets")

# NO importar chatterbox ni torchaudio en el proceso Jupyter hasta terminar.
# El instalador anterior puede haber cambiado torch 2.11 sin cambiar torchvision.
needs_repair = any(package_version(name) != ver for name, ver in PINNED_TORCH.items())
if not needs_repair:
    test = subprocess.run(
        [sys.executable, "-c",
         "import torch, torchaudio, torchvision; "
         "assert torch.cuda.is_available(), 'CUDA no disponible'"],
        capture_output=True, text=True
    )
    needs_repair = test.returncode != 0

if needs_repair:
    print("Corrigiendo el trío GPU a torch 2.6 / torchvision 0.21 / torchaudio 2.6 + CUDA 12.6.")
    run_pip("--force-reinstall", "--no-cache-dir",
            "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0",
            "--index-url", CUDA_WHEELS)

# Instalar la implementación oficial Multilingual V3 sobre el paquete base.
# El parche se instala SIN dependencias para no cambiar de nuevo PyTorch.
import inspect
source = "git+https://github.com/resemble-ai/chatterbox.git@5de7a54aa4e5e2baadb0182dde554908b48b85c2"
installed = metadata.distribution("chatterbox-tts")
direct = installed.read_text("direct_url.json") or ""
if "5de7a54aa4e5e2baadb0182dde554908b48b85c2" not in direct:
    print("Aplicando implementación V3 fijada desde GitHub oficial...")
    run_pip("--force-reinstall", "--no-deps", source)

# Perth 1.0.x imports pkg_resources. Recent setuptools removed it; Perth
# suppresses ImportError and exposes PerthImplicitWatermarker=None instead.
# Keep the real audio watermark active, never substitute DummyWatermarker.
run_pip("setuptools<81")

# Comprobar los imports en un proceso NUEVO: protege de los módulos viejos
# cargados en la memoria del kernel Jupyter.
verify = r'''
import torch, torchvision, torchaudio, inspect, perth
import pkg_resources
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
assert perth.PerthImplicitWatermarker is not None, "Perth no cargó la marca de agua"
watermarker = perth.PerthImplicitWatermarker()  # real pretrained watermark model
print("Marca de agua Perth activa")
assert torch.__version__.split("+")[0] == "2.6.0"
assert torchvision.__version__.split("+")[0] == "0.21.0"
assert torchaudio.__version__.split("+")[0] == "2.6.0"
assert torch.cuda.is_available(), "GPU T4 no disponible"
assert "t3_model" in inspect.signature(ChatterboxMultilingualTTS.from_local).parameters
print("OK:", torch.__version__, torchvision.__version__, torchaudio.__version__)
'''
checked = subprocess.run([sys.executable, "-c", verify], capture_output=True, text=True)
if checked.returncode:
    print(checked.stderr[-3500:])
    raise RuntimeError("Los imports GPU aún fallan. Envía el mensaje final; no cargues los pesos todavía.")
print(checked.stdout.strip())
print("INSTALACIÓN LISTA. Si ejecutaste pip con Torch previamente cargado,")
print("reinicia el kernel desde Run → Restart Session antes de seguir.")

In [ ]:
# Pesos V3 desde ModelScope, no Hugging Face.
from pathlib import Path
import hashlib
MODEL_DIR = Path("/kaggle/working/eduai_chatterbox_v3")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FILES = ["ve.pt", "s3gen.pt", "t3_mtl23ls_v3.safetensors",
         "grapheme_mtl_merged_expanded_v1.json"]
missing = [name for name in FILES if not (MODEL_DIR / name).is_file()]
if missing:
    subprocess.run(["modelscope", "download", "--model", "ResembleAI/chatterbox",
                    *missing, "--local_dir", str(MODEL_DIR)], check=True)
for name in FILES:
    file = MODEL_DIR / name
    assert file.is_file() and file.stat().st_size > 2048, f"Falta: {name}"
print("Archivos verificados presentes:", len(FILES), "· modelo descargado de ModelScope")

In [ ]:
# Sube tu archivo de voz. El notebook debe permanecer PRIVADO.
import ipywidgets as widgets
from IPython.display import display
upload_voice = widgets.FileUpload(accept=".wav,.mp3,.m4a,.ogg", multiple=False)
display(upload_voice)
print("Sube tu muestra (10–30 segundos); luego ejecuta la celda siguiente.")

In [ ]:
# Normalizar la muestra y preparar la voz en GPU. No se imprime ni almacena la voz en outputs.
import wave
from pathlib import Path
entry = upload_voice.value
if not entry:
    raise ValueError("Primero sube tu muestra vocal en la celda anterior.")
item = list(entry.values())[0] if isinstance(entry, dict) else entry[0]
raw_bytes = bytes(item["content"])
if len(raw_bytes) > 12 * 1024 * 1024:
    raise ValueError("La muestra supera los 12 MB.")
original = Path("/kaggle/working/eduai_input_private")
reference = Path("/kaggle/working/eduai_reference_private.wav")
original.write_bytes(raw_bytes)
subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
                "-i", str(original), "-ac", "1", "-ar", "24000",
                "-acodec", "pcm_s16le", str(reference)], check=True)
original.unlink(missing_ok=True)
with wave.open(str(reference), "rb") as w:
    duration = w.getnframes() / w.getframerate()
if not (10 <= duration <= 30):
    reference.unlink(missing_ok=True)
    raise ValueError(f"Muestra: {duration:.1f}s. Debe durar entre 10 y 30 segundos.")
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
model = ChatterboxMultilingualTTS.from_local(ckpt_dir=MODEL_DIR, device="cuda", t3_model="v3")
model.prepare_conditionals(str(reference))
reference.unlink(missing_ok=True)
upload_voice.value = () if not isinstance(upload_voice.value, dict) else {}
print("Referencia preparada en memoria; copia temporal eliminada. Puedes generar varias frases mientras esta sesión siga activa.")

In [ ]:
# Edita aquí lo que deseas que diga tu voz.
TEXTO = "Hola. Esta es una prueba de mi voz autorizada en EDUAI. Hoy aprenderemos algo nuevo."
if not TEXTO.strip() or len(TEXTO) > 550:
    raise ValueError("Ingresa un texto de 1 a 550 caracteres por generación.")

In [ ]:
# Síntesis de voz V3 y WAV listo para importar en EDUAI.
import torchaudio
from pathlib import Path
from IPython.display import FileLink, display
out = Path("/kaggle/working/eduai_voz_kaggle.wav")
with torch.inference_mode():
    waveform = model.generate(TEXTO.strip(), language_id="es")
torchaudio.save(str(out), waveform.detach().cpu(), model.sr, encoding="PCM_S", bits_per_sample=16)
print("Generación completa:", out.name, "· WAV; descárgalo desde Output o el enlace:")
display(FileLink(str(out)))
print("Después, utiliza «Importar WAV de Kaggle» en EDUAI Mis voces.")

### Para volver a usarlo

Con la misma sesión abierta, cambia `TEXTO` y ejecuta las dos últimas celdas. Para una sesión nueva, utiliza **Abrir mi notebook de Kaggle** en EDUAI; selección T4 x2 e Internet, y repite las celdas. El modelo y la voz temporal no sobreviven al reinicio de Kaggle.

**No guardes como versión pública** un notebook con muestras vocales o archivos generados. El resultado se incorpora a EDUAI mediante carga manual autorizada.